# Lesson 19: TensorFlow and Keras 🟧
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Use TensorFlow tensors and `GradientTape` (TensorFlow's autograd)
2. Build, train and evaluate a model in a few lines with **Keras**
3. Compare TensorFlow/Keras with PyTorch and know when people choose each

**⏱️ Timing:** TF basics 10' | Keras digits model 25' | CNN upgrade 15' | PyTorch vs TF 10'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 🚗 An analogy: manual vs automatic car
- **PyTorch** feels like a **manual car**: you write the training loop yourself, so you control every step.
- **Keras** (TensorFlow's high-level API) feels like an **automatic car**: `model.fit()` runs the whole loop for you.

Both get you to the same place. **TensorFlow** was created by Google Brain (released 2015) and is popular in industry and on phones (TensorFlow Lite) and websites (TensorFlow.js). Keras 3 can even run on top of PyTorch or JAX.

In [ ]:
import tensorflow as tf
import numpy as np
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

x = tf.constant([[1., 2.], [3., 4.]])
print(x)
print("matrix multiply:\n", (x @ x).numpy())
print("same as NumPy:\n", np.array([[1, 2], [3, 4]]) @ np.array([[1, 2], [3, 4]]))

In [ ]:
# GradientTape = TensorFlow's autograd: it "records" the maths so it can play it backwards
w = tf.Variable(-4.0)
for step in range(30):
    with tf.GradientTape() as tape:
        loss = (w - 3) ** 2
    grad = tape.gradient(loss, w)
    w.assign_sub(0.1 * grad)
print("ended at w =", round(w.numpy(), 3))

### 🧩 Exercise 1
Compare with Lesson 18's autograd cell. Fill in the table:
| | PyTorch | TensorFlow |
|---|---|---|
| Mark a number to track | `requires_grad=True` | |
| Compute gradients | `loss.backward()` | |
| Read the gradient | `w.grad` | |

## Part 2: Keras — a digit reader in 10 lines ✍️
We'll use the full MNIST dataset (70,000 handwritten digits).

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0          # pixels 0–255 → 0–1
print(x_train.shape, y_train[:10])

model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),                  # 28×28 → 784
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),               # randomly switch off 20% of neurons while training → less memorising
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
history = model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1)   # the whole training loop!
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.2%}")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.show()

probs = model.predict(x_test[:8], verbose=0)
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, p in zip(axes, x_test[:8], probs):
    ax.imshow(img, cmap="gray"); ax.set_title(f"{p.argmax()} ({p.max():.0%})"); ax.axis("off")
plt.show()

### 🧩 Exercise 2
1. If the train accuracy keeps rising but validation accuracy stops — what is happening? (**Overfitting**: memorising instead of learning.)
2. Remove the `Dropout` layer and train 15 epochs. Compare the two curves.

## Part 3: Upgrade to a CNN 🔍
A **Convolutional Neural Network (CNN)** slides small filters across the picture to find edges, curves and shapes — like scanning a page with a magnifying glass. CNNs are the classic choice for images.

In [ ]:
cnn = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(16, 3, activation="relu"),   # 16 filters of size 3×3
    tf.keras.layers.MaxPooling2D(),                     # shrink by half, keep the strongest signals
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation="softmax"),
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(x_train[..., None], y_train, epochs=3, batch_size=64, validation_split=0.1)
print(f"CNN test accuracy: {cnn.evaluate(x_test[..., None], y_test, verbose=0)[1]:.2%}")

In [ ]:
# What do the first-layer filters "see"? Show one digit through all 16 filters
feature_model = tf.keras.Model(cnn.inputs, cnn.layers[0].output)
maps = feature_model.predict(x_test[:1, ..., None], verbose=0)[0]
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(maps[:, :, i], cmap="viridis"); ax.axis("off")
plt.suptitle("16 filters looking at the same digit (edges in different directions)"); plt.show()

In [ ]:
# Save the model — Keras format works on servers; it can also be converted for phones and browsers
cnn.save("digits_cnn.keras")
loaded = tf.keras.models.load_model("digits_cnn.keras")
print("Reloaded model predicts:", loaded.predict(x_test[:5, ..., None], verbose=0).argmax(1), "| true:", y_test[:5])

## Part 4: PyTorch vs TensorFlow — which should I learn? 🤔
| | PyTorch | TensorFlow / Keras |
|---|---|---|
| Made by | Meta AI → PyTorch Foundation | Google |
| Style | You write the loop (flexible, "Pythonic") | `model.fit()` does it (quick to start) |
| Popular in | Research, most new generative AI models, Hugging Face | Industry production, mobile (TF Lite), browser (TF.js) |
| Same ideas? | ✅ tensors, autograd, layers, optimisers — the concepts transfer directly |

### 🚀 Challenge
Rebuild the Keras dense model **in PyTorch** using the training loop from Lesson 18. Which version needed more code? Which one helped you understand more?

## 📌 Summary
- TensorFlow tensors + `GradientTape` = PyTorch tensors + autograd
- **Keras**: `Sequential` → `compile` → `fit` → `evaluate` → `predict`
- **CNNs** use sliding filters and work great for images
- Learn the **concepts**; switching frameworks is then easy